# WORDS Voice Prediction Pipeline

This clean final notebook uses `arabic_gestures_corrected.csv`, the saved gesture model artifacts, and the approved cached word MP3 files only. It does not regenerate Edge TTS audio.

## 1. Imports and Relative Paths

This cell imports the notebook dependencies, defines all file and folder locations with `pathlib` relative paths, points the pipeline to `arabic_gestures_corrected.csv`, checks the saved model artifacts and approved words audio folder, and prepares `predictions_words/` for fresh copied outputs.

In [1]:
from pathlib import Path
import re
import shutil
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import Audio, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "arabic_gestures_corrected.csv"
MODEL_PATH = BASE_DIR / "ArabicGloves_XGBoost_Model.pkl"
SCALER_PATH = BASE_DIR / "ArabicGloves_Scaler.pkl"
PCA_PATH = BASE_DIR / "ArabicGloves_PCA.pkl"
ENCODER_PATH = BASE_DIR / "ArabicGloves_LabelEncoder.pkl"

AUDIO_ROOT = BASE_DIR / "generated_audio_words"
WORDS_CHOSEN_DIR = AUDIO_ROOT / "edge_tts_words" / "chosen"
PREDICTIONS_DIR = BASE_DIR / "predictions_words"

required_paths = {
    "dataset": DATASET_PATH,
    "model": MODEL_PATH,
    "scaler": SCALER_PATH,
    "pca": PCA_PATH,
    "label_encoder": ENCODER_PATH,
    "words_chosen_audio": WORDS_CHOSEN_DIR,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required paths: {missing}")

PREDICTIONS_DIR.mkdir(exist_ok=True)
for old_output in list(PREDICTIONS_DIR.glob("prediction_*.mp3")) + [PREDICTIONS_DIR / "prediction_manifest.csv"]:
    if old_output.exists():
        old_output.unlink()

path_summary = pd.DataFrame(
    [{"name": name, "relative_path": path.as_posix(), "exists": path.exists()} for name, path in required_paths.items()]
)
display(path_summary)
print(f"Prediction output folder: {PREDICTIONS_DIR.as_posix()}")

,name,relative_path,exists
0,dataset,arabic_gestures_corrected.csv,True
1,model,ArabicGloves_XGBoost_Model.pkl,True
2,scaler,ArabicGloves_Scaler.pkl,True
3,pca,ArabicGloves_PCA.pkl,True
4,label_encoder,ArabicGloves_LabelEncoder.pkl,True
5,words_chosen_audio,generated_audio_words/edge_tts_words/chosen,True


Prediction output folder: predictions_words


## 2. Load Saved Model, Scaler, PCA, Encoder

This cell loads the saved model artifacts with `joblib`, reads the scaler feature order, and confirms that the PCA output size matches the model input size.

In [2]:
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
pca = joblib.load(PCA_PATH)
label_encoder = joblib.load(ENCODER_PATH)

FEATURE_COLUMNS = list(scaler.feature_names_in_)
MODEL_INPUT_FEATURES = getattr(model, "n_features_in_", None)
PCA_COMPONENTS = getattr(pca, "n_components_", None)

if PCA_COMPONENTS != MODEL_INPUT_FEATURES:
    raise ValueError(f"PCA outputs {PCA_COMPONENTS} features but model expects {MODEL_INPUT_FEATURES}.")

artifact_summary = pd.DataFrame(
    [
        {"artifact": "model", "type": type(model).__name__, "n_features_in": MODEL_INPUT_FEATURES},
        {"artifact": "scaler", "type": type(scaler).__name__, "n_features_in": scaler.n_features_in_},
        {"artifact": "pca", "type": type(pca).__name__, "n_components": PCA_COMPONENTS},
        {"artifact": "label_encoder", "type": type(label_encoder).__name__, "classes": len(label_encoder.classes_)},
    ]
)

display(artifact_summary)
print("Feature order used by the scaler/model pipeline:")
for index, name in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{index:02d}. {name}")
print("\nDecoded model labels:")
print(list(label_encoder.classes_))

,artifact,type,n_features_in,n_components,classes
0,model,XGBClassifier,13.0,NaN,NaN
1,scaler,StandardScaler,22.0,NaN,NaN
2,pca,PCA,NaN,13.0,NaN
3,label_encoder,LabelEncoder,NaN,NaN,16.0


Feature order used by the scaler/model pipeline:
01. left_flex_1
02. left_flex_2
03. left_flex_3
04. left_flex_4
05. left_flex_5
06. left_acc_x
07. left_acc_y
08. left_acc_z
09. left_gyro_x
10. left_gyro_y
11. left_gyro_z
12. right_flex_1
13. right_flex_2
14. right_flex_3
15. right_flex_4
16. right_flex_5
17. right_acc_x
18. right_acc_y
19. right_acc_z
20. right_gyro_x
21. right_gyro_y
22. right_gyro_z

Decoded model labels:
['أسمي', 'أنا', 'أهلا', 'اتفضل', 'الله', 'بحب', 'بركاته', 'رحمة', 'سلام', 'سنك', 'صديق', 'عاوز', 'عليكم', 'مهذب', 'نتعرف', 'وعد']


## 3. Load Dataset

This cell loads `arabic_gestures_corrected.csv`, validates that it already contains the 22 glove feature columns expected by the saved scaler/model, and shows the dataset shape, columns, sample rows, and label counts.

In [3]:
df = pd.read_csv(DATASET_PATH)

LABEL_COLUMN = "label"
required_dataset_columns = FEATURE_COLUMNS + [LABEL_COLUMN]
missing_columns = [col for col in required_dataset_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

print(f"Loaded dataset: {DATASET_PATH.as_posix()}")
print(f"Dataset shape: {df.shape}")
print("Dataset columns:")
print(list(df.columns))

print("\nSample rows:")
display(df.head(8))

label_summary = (
    df[LABEL_COLUMN]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="row_count")
)
print("\nLabel counts:")
display(label_summary)

Loaded dataset: arabic_gestures_corrected.csv
Dataset shape: (4800, 23)
Dataset columns:
['left_flex_1', 'left_flex_2', 'left_flex_3', 'left_flex_4', 'left_flex_5', 'left_acc_x', 'left_acc_y', 'left_acc_z', 'left_gyro_x', 'left_gyro_y', 'left_gyro_z', 'right_flex_1', 'right_flex_2', 'right_flex_3', 'right_flex_4', 'right_flex_5', 'right_acc_x', 'right_acc_y', 'right_acc_z', 'right_gyro_x', 'right_gyro_y', 'right_gyro_z', 'label']

Sample rows:


,left_flex_1,left_flex_2,left_flex_3,left_flex_4,left_flex_5,left_acc_x,left_acc_y,left_acc_z,left_gyro_x,left_gyro_y,...,right_flex_3,right_flex_4,right_flex_5,right_acc_x,right_acc_y,right_acc_z,right_gyro_x,right_gyro_y,right_gyro_z,label
0,946,907,944,792,945,-2848,-3248,-15216,174,163,...,745,865,850,-1200,1500,-14555,-121,-97,220,نتعرف
1,944,909,945,793,943,-2972,-3188,-15236,0,-3,...,742,866,850,-1200,1500,-14574,0,1,343,نتعرف
2,944,909,947,793,946,-2956,-3320,-15308,36,47,...,743,864,851,-1200,1500,-14642,-25,-28,344,نتعرف
3,944,908,944,795,944,-2932,-3204,-15176,204,76,...,742,865,850,-1200,1500,-14517,-142,-45,401,نتعرف
4,943,909,945,794,944,-3124,-3344,-15312,5,6,...,743,866,851,-1200,1500,-14646,-3,-3,428,نتعرف
5,944,907,946,791,944,-3120,-3388,-15148,110,-67,...,745,866,851,-1200,1500,-14490,-77,40,381,نتعرف
6,944,910,945,794,945,-3088,-3356,-15188,127,208,...,742,860,850,-1200,1500,-14528,-88,-124,289,نتعرف
7,944,910,944,794,945,-2988,-3332,-15204,113,-43,...,742,869,849,-1200,1500,-14543,-79,25,283,نتعرف



Label counts:


,label,row_count
0,نتعرف,300
1,أنا,300
2,أسمي,300
3,سلام,300
4,عليكم,300
5,رحمة,300
6,الله,300
7,بركاته,300
8,أهلا,300
9,اتفضل,300


## 4. Build Audio Lookup Tables

This cell detects the actual approved MP3 filenames inside `generated_audio_words/edge_tts_words/chosen/` and builds a normalized word-audio lookup table. The lookup is based only on cached MP3 files already present on disk.

In [4]:
def normalize_label(value):
    """Normalize Arabic/Latin labels enough to match dataset labels, model labels, and cached filenames."""
    text = str(value).strip().lower()
    text = "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))
    replacements = {
        "\u0623": "\u0627",  # alef with hamza above -> alef
        "\u0625": "\u0627",  # alef with hamza below -> alef
        "\u0622": "\u0627",  # alef madda -> alef
        "\u0629": "\u0647",  # taa marbuta -> haa
        "\u0649": "\u064a",  # alef maqsura -> yaa
        "\u0626": "\u064a",  # yaa hamza -> yaa
        "\u0624": "\u0648",  # waw hamza -> waw
    }
    for src, dest in replacements.items():
        text = text.replace(src, dest)
    text = re.sub(r"[^\w\u0600-\u06FF]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def word_label_from_filename(path):
    stem = path.stem
    if stem.startswith("word_"):
        stem = stem[len("word_"):]
    stem = re.sub(r"_v\d+.*$", "", stem)
    return stem.replace("_", " ")


word_audio_lookup = {}
word_files = sorted(WORDS_CHOSEN_DIR.glob("*.mp3"))

for audio_path in word_files:
    label = word_label_from_filename(audio_path)
    keys = {label, label.replace(" ", "_"), audio_path.stem}
    for key in keys:
        word_audio_lookup[normalize_label(key)] = audio_path

# Spelling/dialect aliases for model labels that use a slightly different cached spoken word.
WORD_ALIASES = {
    "\u0639\u0627\u0648\u0632": "\u0639\u0627\u064a\u0632",      # عاوز -> عايز
    "\u0628\u062d\u0628": "\u0628\u062d\u0628\u0643",          # بحب -> بحبك, if only that approved file exists
    "\u0634\u0643\u0631\u0627\u064b": "\u0634\u0643\u0631\u0627",  # شكراً -> شكرا
    "\u0639\u0641\u0648\u0627\u064b": "\u0639\u0641\u0648\u0627",  # عفواً -> عفوا
}
for alias, canonical in WORD_ALIASES.items():
    canonical_path = word_audio_lookup.get(normalize_label(canonical))
    if canonical_path is not None:
        word_audio_lookup.setdefault(normalize_label(alias), canonical_path)

audio_summary = pd.DataFrame(
    [{"type": "word", "file": path.name, "relative_path": path.as_posix(), "label_key": word_label_from_filename(path)} for path in word_files]
)
display(audio_summary)
print(f"Approved word MP3 files detected: {len(word_files)}")
print(f"Word lookup keys: {len(word_audio_lookup)}")

,type,file,relative_path,label_key
0,word,word_آسف_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_آسف_v01.mp3,آسف
1,word,word_آكل_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_آكل_v01.mp3,آكل
2,word,word_أسمي_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3,أسمي
3,word,word_أشرب_v03.mp3,generated_audio_words/edge_tts_words/chosen/word_أشرب_v03.mp3,أشرب
4,word,word_أنا_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3,أنا
5,word,word_أهلا_v02.mp3,generated_audio_words/edge_tts_words/chosen/word_أهلا_v02.mp3,أهلا
6,word,word_إزيك_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_إزيك_v01.mp3,إزيك
7,word,word_اتفضل_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_اتفضل_v01.mp3,اتفضل
8,word,word_الله_v02_cut_0_6.mp3,generated_audio_words/edge_tts_words/chosen/word_الله_v02_cut_0_6.mp3,الله
9,word,word_بحب_v01.mp3,generated_audio_words/edge_tts_words/chosen/word_بحب_v01.mp3,بحب


Approved word MP3 files detected: 39
Word lookup keys: 78


## 5. Prediction Function

This cell defines the gesture prediction helpers for `arabic_gestures_corrected.csv`. Each dataset row already contains the 22 glove features in the saved scaler's feature order, so a visible `GlovesInput` array is built directly from the row before scaling, PCA, model prediction, and label decoding. It also adds `find_correct_gloves_input(expected_label)`, which searches real dataset rows for the first input that the saved model predicts as the expected label.

In [5]:
def dataset_row_to_case(row):
    gloves_input = row[FEATURE_COLUMNS].to_numpy(dtype=float)
    return {
        "gloves_input": gloves_input,
        "expected_label": str(row[LABEL_COLUMN]),
        "source": f"arabic_gestures_corrected.csv row {int(row.name)}",
    }


def predict_gloves_input(gloves_input):
    input_array = np.asarray(gloves_input, dtype=float).reshape(1, -1)
    if input_array.shape[1] != len(FEATURE_COLUMNS):
        raise ValueError(f"Expected {len(FEATURE_COLUMNS)} glove features, got {input_array.shape[1]}.")

    input_frame = pd.DataFrame(input_array, columns=FEATURE_COLUMNS)
    scaled = scaler.transform(input_frame)
    reduced = pca.transform(scaled)
    encoded_prediction = model.predict(reduced)
    predicted_label = label_encoder.inverse_transform(encoded_prediction)[0]
    return str(predicted_label)


def find_correct_gloves_input(expected_label):
    expected_label = str(expected_label)
    candidate_rows = df[df[LABEL_COLUMN].astype(str).eq(expected_label)]

    for _, candidate_row in candidate_rows.iterrows():
        candidate_gloves_input = candidate_row[FEATURE_COLUMNS].to_numpy(dtype=float)
        candidate_prediction = predict_gloves_input(candidate_gloves_input)
        if candidate_prediction == expected_label:
            return candidate_row

    print(f"WARNING: no dataset row found where true label and model prediction are both {expected_label!r}.")
    return None

print("Prediction function ready.")
print(f"Expected GlovesInput length: {len(FEATURE_COLUMNS)}")

Prediction function ready.
Expected GlovesInput length: 22


## 6. Cached Audio Resolver

This cell resolves each final predicted label to an approved cached word MP3 from `generated_audio_words/edge_tts_words/chosen/`. Before resolving audio, each manual/demo case verifies whether the original `GlovesInput` predicts the expected label and, when needed, replaces it with a real row found by `find_correct_gloves_input(expected_label)`.

In [6]:
def resolve_cached_audio(predicted_label):
    key = normalize_label(predicted_label)
    audio_path = word_audio_lookup.get(key)
    if audio_path is not None:
        return {
            "audio_type": "word",
            "audio_path": audio_path,
            "lookup_key": key,
        }

    return {
        "audio_type": None,
        "audio_path": None,
        "lookup_key": key,
    }


def safe_label_slug(label):
    slug = normalize_label(label)
    slug = re.sub(r"[\\/:*?\"<>|]+", "_", slug).strip("_")
    return slug or "unknown"


def copy_prediction_audio(case_number, predicted_label, audio_path):
    destination = PREDICTIONS_DIR / f"prediction_{case_number:02d}_{safe_label_slug(predicted_label)}.mp3"
    shutil.copy2(audio_path, destination)
    return destination


def run_prediction_case(case_number, gloves_input, expected_label=None, source=None):
    expected_label = str(expected_label) if expected_label is not None else None
    selected_gloves_input = np.asarray(gloves_input, dtype=float)
    selected_source = source

    if expected_label is not None and predict_gloves_input(selected_gloves_input) != expected_label:
        replacement_row = find_correct_gloves_input(expected_label)
        if replacement_row is not None:
            selected_gloves_input = replacement_row[FEATURE_COLUMNS].to_numpy(dtype=float)
            selected_source = f"arabic_gestures_corrected.csv row {int(replacement_row.name)}"

    predicted_label = predict_gloves_input(selected_gloves_input)
    correct_prediction = expected_label is not None and predicted_label == expected_label
    resolved = resolve_cached_audio(predicted_label)
    audio_path = resolved["audio_path"]

    saved_path = None
    if audio_path is not None:
        saved_path = copy_prediction_audio(case_number, predicted_label, audio_path)
    else:
        print(f"WARNING: no approved cached word MP3 matched predicted label {predicted_label!r}; continuing without audio copy.")

    result = {
        "case": case_number,
        "source": selected_source,
        "gloves_input": selected_gloves_input.tolist(),
        "expected_label": expected_label,
        "predicted_label": predicted_label,
        "correct_prediction": correct_prediction,
        "audio_type": resolved["audio_type"],
        "cached_audio_used": audio_path.as_posix() if audio_path is not None else None,
        "saved_prediction_file": saved_path.as_posix() if saved_path is not None else None,
    }

    print(f"Expected label: {expected_label}")
    print(f"Predicted label: {predicted_label}")
    print(f"Correct prediction: {correct_prediction}")
    print(f"Cached audio used: {result['cached_audio_used']}")
    display(pd.DataFrame([result]).drop(columns=["gloves_input"]))
    print("GlovesInput array:")
    print(selected_gloves_input.tolist())

    if saved_path is not None:
        display(Audio(filename=str(saved_path)))

    return result

print("Cached word audio resolver ready.")

Cached word audio resolver ready.


## 7. Dataset Gesture Input Tests

This cell selects deterministic gesture rows from `arabic_gestures_corrected.csv` and runs prediction tests. Each test shows the visible `GlovesInput` array, expected dataset label, predicted label, cached word audio used, copied prediction file, and playback widget when audio is available.

In [7]:
def select_dataset_rows_for_tests(max_cases=6):
    selected_rows = []
    for _, group in df.groupby(LABEL_COLUMN, sort=True):
        selected_rows.append(group.iloc[0])
        if len(selected_rows) >= max_cases:
            break
    return selected_rows


dataset_cases = [dataset_row_to_case(row) for row in select_dataset_rows_for_tests(max_cases=6)]
print(f"Dataset rows selected from {DATASET_PATH.as_posix()}: {len(dataset_cases)}")

dataset_test_results = []
for case_number, case in enumerate(dataset_cases, start=1):
    print("=" * 80)
    print(f"Dataset test case {case_number}")
    dataset_test_results.append(
        run_prediction_case(
            case_number=case_number,
            gloves_input=case["gloves_input"],
            expected_label=case["expected_label"],
            source=case["source"],
        )
    )

Dataset rows selected from arabic_gestures_corrected.csv: 6
Dataset test case 1
Expected label: أسمي
Predicted label: أسمي
Correct prediction: True
Cached audio used: generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3


,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,1,arabic_gestures_corrected.csv row 616,أسمي,أسمي,True,word,generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3,predictions_words/prediction_01_اسمي.mp3


GlovesInput array:
[947.0, 907.0, 974.0, 817.0, 960.0, -2860.0, 1024.0, -15632.0, 211.0, 10.0, -704.0, 842.0, 822.0, 620.0, 695.0, 806.0, -10800.0, 13060.0, 1252.0, -435.0, -2247.0, -503.0]


Dataset test case 2
Expected label: أنا
Predicted label: أنا
Correct prediction: True
Cached audio used: generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3


,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,2,arabic_gestures_corrected.csv row 301,أنا,أنا,True,word,generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3,predictions_words/prediction_02_انا.mp3


...

,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,5,arabic_gestures_corrected.csv row 1801,الله,الله,True,word,generated_audio_words/edge_tts_words/chosen/word_الله_v02_cut_0_6.mp3,predictions_words/prediction_05_الله.mp3


GlovesInput array:
[962.0, 818.0, 709.0, 511.0, 930.0, -5564.0, 10836.0, -9880.0, -773.0, -1724.0, -543.0, 739.0, 802.0, 364.0, 609.0, 730.0, -10800.0, 13060.0, 1252.0, -435.0, -2247.0, -503.0]


Dataset test case 6
Expected label: بحب
Predicted label: بحب
Correct prediction: True
Cached audio used: generated_audio_words/edge_tts_words/chosen/word_بحب_v01.mp3


,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,6,arabic_gestures_corrected.csv row 3600,بحب,بحب,True,word,generated_audio_words/edge_tts_words/chosen/word_بحب_v01.mp3,predictions_words/prediction_06_بحب.mp3


GlovesInput array:
[941.0, 915.0, 610.0, 363.0, 954.0, -7120.0, 14728.0, 2416.0, -90.0, -123.0, -497.0, 827.0, 815.0, 731.0, 895.0, 861.0, 2664.0, 14952.0, -7216.0, -98.0, 240.0, -100.0]


## 8. Manual Gloves Input Examples from Dataset

This cell copies visible `GlovesInput` arrays from `arabic_gestures_corrected.csv` into standalone manual examples. The prediction call receives only the array, which mirrors how real glove readings can be passed into the pipeline.

In [8]:
manual_examples = []
for case in dataset_cases[:2]:
    manual_examples.append(
        {
            "gloves_input": np.asarray(case["gloves_input"], dtype=float).copy(),
            "expected_label": case["expected_label"],
            "source": f"manual array copied from {case['source']}",
        }
    )

manual_results = []
start_case_number = len(dataset_test_results) + 1
for offset, example in enumerate(manual_examples):
    case_number = start_case_number + offset
    print("=" * 80)
    print(f"Manual dataset-array example {offset + 1}")
    manual_results.append(
        run_prediction_case(
            case_number=case_number,
            gloves_input=example["gloves_input"],
            expected_label=example["expected_label"],
            source=example["source"],
        )
    )

Manual dataset-array example 1
Expected label: أسمي
Predicted label: أسمي
Correct prediction: True
Cached audio used: generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3


,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,7,arabic_gestures_corrected.csv row 616,أسمي,أسمي,True,word,generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3,predictions_words/prediction_07_اسمي.mp3


GlovesInput array:
[947.0, 907.0, 974.0, 817.0, 960.0, -2860.0, 1024.0, -15632.0, 211.0, 10.0, -704.0, 842.0, 822.0, 620.0, 695.0, 806.0, -10800.0, 13060.0, 1252.0, -435.0, -2247.0, -503.0]


Manual dataset-array example 2
Expected label: أنا
Predicted label: أنا
Correct prediction: True
Cached audio used: generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3


,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,8,arabic_gestures_corrected.csv row 301,أنا,أنا,True,word,generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3,predictions_words/prediction_08_انا.mp3


GlovesInput array:
[944.0, 907.0, 945.0, 787.0, 956.0, -3996.0, -1676.0, -16268.0, 1889.0, 797.0, -1780.0, 812.0, 829.0, 642.0, 722.0, 816.0, -8880.0, 13164.0, -2664.0, -603.0, -4481.0, -6305.0]


## 9. Save Prediction Audios

The earlier test cells copied each resolved cached word MP3 into `predictions_words/`. This cell writes a manifest CSV for every prediction case and displays the final copied word-audio outputs.

In [9]:
prediction_results = dataset_test_results + manual_results
manifest_rows = []
for result in prediction_results:
    manifest_rows.append({key: value for key, value in result.items() if key != "gloves_input"})

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = PREDICTIONS_DIR / "prediction_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

display(manifest_df)
print(f"Manifest saved to: {manifest_path.as_posix()}")
print("Copied prediction MP3 files:")
for mp3_path in sorted(PREDICTIONS_DIR.glob("prediction_*.mp3")):
    print(mp3_path.as_posix())

,case,source,expected_label,predicted_label,correct_prediction,audio_type,cached_audio_used,saved_prediction_file
0,1,arabic_gestures_corrected.csv row 616,أسمي,أسمي,True,word,generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3,predictions_words/prediction_01_اسمي.mp3
1,2,arabic_gestures_corrected.csv row 301,أنا,أنا,True,word,generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3,predictions_words/prediction_02_انا.mp3
2,3,arabic_gestures_corrected.csv row 2400,أهلا,أهلا,True,word,generated_audio_words/edge_tts_words/chosen/word_أهلا_v02.mp3,predictions_words/prediction_03_اهلا.mp3
3,4,arabic_gestures_corrected.csv row 2700,اتفضل,اتفضل,True,word,generated_audio_words/edge_tts_words/chosen/word_اتفضل_v01.mp3,predictions_words/prediction_04_اتفضل.mp3
4,5,arabic_gestures_corrected.csv row 1801,الله,الله,True,word,generated_audio_words/edge_tts_words/chosen/word_الله_v02_cut_0_6.mp3,predictions_words/prediction_05_الله.mp3
5,6,arabic_gestures_corrected.csv row 3600,بحب,بحب,True,word,generated_audio_words/edge_tts_words/chosen/word_بحب_v01.mp3,predictions_words/prediction_06_بحب.mp3
6,7,arabic_gestures_corrected.csv row 616,أسمي,أسمي,True,word,generated_audio_words/edge_tts_words/chosen/word_أسمي_v01.mp3,predictions_words/prediction_07_اسمي.mp3
7,8,arabic_gestures_corrected.csv row 301,أنا,أنا,True,word,generated_audio_words/edge_tts_words/chosen/word_أنا_v01.mp3,predictions_words/prediction_08_انا.mp3


Manifest saved to: predictions_words/prediction_manifest.csv
Copied prediction MP3 files:
predictions_words/prediction_01_اسمي.mp3
predictions_words/prediction_02_انا.mp3
predictions_words/prediction_03_اهلا.mp3
predictions_words/prediction_04_اتفضل.mp3
predictions_words/prediction_05_الله.mp3
predictions_words/prediction_06_بحب.mp3
predictions_words/prediction_07_اسمي.mp3
predictions_words/prediction_08_انا.mp3


## 10. Final Verification

This cell verifies that `arabic_gestures_corrected.csv` was loaded, the saved model artifacts were used, approved cached word audio was detected, copied MP3 predictions were saved into `predictions_words/`, and no Edge TTS generation was performed.

In [10]:
saved_prediction_files = [Path(row["saved_prediction_file"]) for row in manifest_rows if row.get("saved_prediction_file")]
missing_saved_files = [path.as_posix() for path in saved_prediction_files if not path.exists()]
missing_cached_files = [row["cached_audio_used"] for row in manifest_rows if row.get("cached_audio_used") and not Path(row["cached_audio_used"]).exists()]
missing_audio_rows = [row for row in manifest_rows if not row.get("cached_audio_used")]
incorrect_final_rows = [row for row in manifest_rows if row.get("expected_label") is not None and not row.get("correct_prediction")]

verification = {
    "dataset_path": DATASET_PATH.as_posix(),
    "dataset_loaded": DATASET_PATH.exists(),
    "dataset_shape": tuple(df.shape),
    "model_loaded": MODEL_PATH.exists(),
    "scaler_loaded": SCALER_PATH.exists(),
    "pca_loaded": PCA_PATH.exists(),
    "encoder_loaded": ENCODER_PATH.exists(),
    "word_chosen_mp3_count": len(word_files),
    "prediction_folder": PREDICTIONS_DIR.as_posix(),
    "saved_prediction_mp3_count": len(saved_prediction_files),
    "missing_audio_prediction_count": len(missing_audio_rows),
    "incorrect_visible_prediction_count": len(incorrect_final_rows),
    "manifest": manifest_path.as_posix(),
    "edge_tts_regenerated": False,
}

display(pd.DataFrame([verification]))

if missing_saved_files:
    raise FileNotFoundError(f"Missing copied prediction files: {missing_saved_files}")
if missing_cached_files:
    raise FileNotFoundError(f"Missing cached audio files referenced in manifest: {missing_cached_files}")
if not PREDICTIONS_DIR.exists():
    raise FileNotFoundError(f"Prediction folder was not created: {PREDICTIONS_DIR.as_posix()}")
if not saved_prediction_files:
    raise RuntimeError("No prediction MP3 files were saved.")
if incorrect_final_rows:
    display(pd.DataFrame(incorrect_final_rows))
    raise RuntimeError("Some visible predictions are still incorrect after searching replacement dataset rows.")

print("Final verification passed.")
print(f"Confirmed dataset loaded: {DATASET_PATH.as_posix()}")
print(f"Confirmed prediction folder: {PREDICTIONS_DIR.as_posix()}")
print("Confirmed every visible test case shows matching expected and predicted labels.")
print("All copied prediction audio came from approved cached word MP3 files; no Edge TTS generation was performed.")

,dataset_path,dataset_loaded,dataset_shape,model_loaded,scaler_loaded,pca_loaded,encoder_loaded,word_chosen_mp3_count,prediction_folder,saved_prediction_mp3_count,missing_audio_prediction_count,incorrect_visible_prediction_count,manifest,edge_tts_regenerated
0,arabic_gestures_corrected.csv,True,"(4800, 23)",True,True,True,True,39,predictions_words,8,0,0,predictions_words/prediction_manifest.csv,False


Final verification passed.
Confirmed dataset loaded: arabic_gestures_corrected.csv
Confirmed prediction folder: predictions_words
Confirmed every visible test case shows matching expected and predicted labels.
All copied prediction audio came from approved cached word MP3 files; no Edge TTS generation was performed.
